# 02613 — Exam Cheat Sheet from the Reading Materials
## Based on *Fast Python* (Antão) and *Advanced Python Programming* (Zaccone)

---
**Topics:**
1. [Profiling — cProfile & line_profiler](#1-profiling)
2. [NumPy — Views, Strides & Broadcasting](#2-numpy)
3. [Cache, Memory Hierarchy & Hardware](#3-cache)
4. [Data Compression — Blosc & Efficient Storage](#4-blosc)
5. [Pandas — dtype Optimisation & Indexing](#5-pandas)
6. [Zarr — Chunked On-Disk Arrays](#6-zarr)
7. [Amdahl's Law (Advanced Python Ch. 9)](#7-amdahl)
8. [The GIL — Threading vs Multiprocessing (Advanced Python Ch. 22)](#8-gil)
9. [Parallel Reduction (Advanced Python Ch. 14)](#9-reduction)
10. [Multiprocessing Patterns (Advanced Python Ch. 7 & 13)](#10-multiprocessing)
11. [GPU & CUDA (Fast Python Ch. 9)](#11-gpu)
12. [Numba JIT (Fast Python App. B)](#12-numba)

---

# 1. Profiling — cProfile & line_profiler
### *Fast Python Ch. 2 | Advanced Python Ch. 1*

---

## The Golden Rule
> **"Profile first, optimise second."**  Never guess what is slow. Always measure.

---

## cProfile — Function-level profiler (built-in)

```bash
# Run from command line — sort by cumulative time
python -m cProfile -s cumulative myscript.py input.csv

# Save to file for snakeviz
python -m cProfile -o output.prof myscript.py
snakeviz output.prof
```

### Reading cProfile output

```
ncalls  tottime  percall  cumtime  percall  filename:function
    10    0.000    0.000    5.055    0.505  script.py:process_sample
     1    0.000    0.000   20.009   20.009  script.py:load_params
```

| Column | Meaning | Use for |
|---|---|---|
| `ncalls` | Number of calls | Infer data size (calls per sample) |
| `tottime` | Time in this function only | Own code bottleneck |
| `cumtime` | Total time incl. sub-calls | **Overall impact** ← use this |
| `percall` | cumtime / ncalls | Scales to production workload |

### Exam pattern: find production bottleneck
```
process_sample: percall=0.505s, ncalls=10 (profiling subset)
→ production (1000 samples): 1000 × 0.505 = 505s  ← FOCUS HERE

load_params: cumtime=20s, ncalls=1 (called once, does not scale)
→ production: still just 20s
```

---

## line_profiler — Line-by-line profiler

```bash
pip install line_profiler
kernprof -l -v script.py
```

```python
@profile          # decorator marks the function to profile
def my_function():
    ...
```

### Reading line_profiler output

```
Timer unit: 1e-06 s
Line #  Hits    Time   Per Hit  % Time  Line Contents
    8  10000  5000.0      0.5    33.3   a = x[i] * x[i] + 4
    9  10000  5000.0      0.5    33.3   b = y[n-i-1] / x[i]
   10  10000  3000.0      0.3    20.0   z = z + a / b
```

| Column | Meaning |
|---|---|
| `Hits` | Times this line ran = n_iterations |
| `Time` | Total time for this line (in Timer units, usually µs) |
| `% Time` | **Focus on the highest values** |

### FLOP/s calculation from line_profiler
1. Count arithmetic ops per iteration: `+`, `-`, `*`, `/`, `sqrt`, `sin`, `cos` = 1 FLOP each
2. Total FLOPs = FLOPs/iter × Hits
3. Total time (s) = sum of Time column ÷ 1,000,000
4. FLOP/s = Total FLOPs / Total time (s)

In [1]:
import cProfile, pstats, io

# ── FLOP/s calculator ────────────────────────────────────────────
def flops_per_second(flops_per_iter, n_iters, total_time_us):
    total_flops  = flops_per_iter * n_iters
    total_time_s = total_time_us / 1e6
    return total_flops / total_time_s

# Re-exam 2024 Q8 example:
# Line 8:  a = x[i]*x[i]+4    → 2 FLOPs (mul + add)
# Line 9:  b = y[n-i-1]/x[i]  → 1 FLOP  (div)
# Line 10: z = z + a/b         → 2 FLOPs (div + add)
# 5 FLOPs/iter × 10000 iters, total time = 2000+5000+5000+3000 = 15000 µs
result = flops_per_second(5, 10000, 15000)
print(f"FLOP/s = {result:.3e}")  # ~3.33e6

# ── Production bottleneck estimator ─────────────────────────────
profile = [
    ('load_params',    1,   20.009, False),  # (name, ncalls, percall, scales)
    ('process_sample', 10,  0.505, True),
    ('save',           1,   3.007, False),
]
n_production = 1000
print(f"\n{'Function':<20} {'prod time':>12}")
for name, ncalls, percall, scales in profile:
    prod_time = percall * (n_production if scales else 1)
    flag = " ← BOTTLENECK" if prod_time == max(p*(n_production if s else 1) for _,_,p,s in profile) else ""
    print(f"{name:<20} {prod_time:>11.1f}s{flag}")

FLOP/s = 3.333e+06

Function                prod time
load_params                 20.0s
process_sample             505.0s ← BOTTLENECK
save                         3.0s


---
# 2. NumPy — Views, Strides & Broadcasting
### *Fast Python Ch. 4*

---

## Views vs Copies

| | View | Copy |
|---|---|---|
| Memory | **Shared** with original | Independent |
| Speed | Fast (no data copy) | Slower (data copy) |
| Memory use | No extra | Doubles memory |
| Changing view | **Modifies original too!** | Safe |

```python
import numpy as np

arr = np.arange(10)
view = arr[::-1]        # view — shares memory, no copy
copy = arr[::-1].copy() # copy — independent

# Check if shared:
np.shares_memory(arr, view)  # True
view.base is arr             # True (for direct views)

# These operations return VIEWS (fast, memory-efficient):
# reshape, T (transpose), [::-1], slicing with step, flipud/fliplr

# These ALWAYS return COPIES (fancy indexing):
# arr[[0, 2, 4]]         ← list of indices
# arr[arr > 5]           ← boolean mask
# arr.reshape(7)         ← if not contiguous
```

---

## Strides — the key to understanding memory layout

**Stride** = how many **bytes** to skip to reach the next element along that axis.

```python
import numpy as np

linear = np.arange(10, dtype=np.uint32)  # 4 bytes each
# linear.strides = (4,)  ← 4 bytes to next element

m2x5 = linear.reshape(2, 5)
# m2x5.strides = (20, 4)  ← 20 bytes to next row, 4 to next col

m5x2 = m2x5.T
# m5x2.strides = (4, 20)  ← transposed strides!
```

### Rule for cache-efficient loops
> **Innermost loop = smallest stride** (sequential memory access → no cache misses)

```python
# Given strides (600, 40, 8, 200) for axes (i, j, k, l):
# Smallest stride = 8 at axis k → k must be innermost
# Order outer→inner: i(600) → l(200) → j(40) → k(8)

for i in ...:      # stride 600
  for l in ...:    # stride 200
    for j in ...:  # stride 40
      for k in ...: # stride 8 ← innermost
          x = arr[i, j, k, l]
```

---

## Broadcasting — the 4-step rule

1. **Align shapes to the right**
2. **Left-pad with 1s** until same length
3. **Check**: each dim must be equal OR one of them is 1
4. **Result shape** = element-wise max

```
a: (100, 1, 6, 3)     b: (100, 1, 3)
→ pad b: (1, 100, 1, 3)
→ broadcast: (100, 100, 6, 3)
```

### Adding dimensions with None
```python
a[:, None]       # adds axis after dim 0: (n,) → (n,1)
a[None, :]       # adds axis at front:    (n,) → (1,n)
a[:, :, None]    # adds axis after dim 1
a[..., None]     # adds axis at end
```

### Common exam patterns
```python
# images (N,H,W,3) - subtract mean image (H,W)
images - mim[:, :, None]          # mim→(H,W,1) broadcasts over 3 channels

# images (N,H,W,3) - subtract per-image mean pixel (N,3)
images - mean_pixels[:, None, None, :]  # →(N,1,1,3) broadcasts over H,W

# Outer product: x(n,) y(m,) → (n,m)
x[:, None] * y[None, :]           # (n,1)*(1,m) → (n,m)

# Distance matrix: D[i,j] = |x[i] - y[j]|
np.abs(x[:, None] - y[None, :])   # (n,1)-(1,m) → (n,m)

# Standardise rows: data(n,d), mean(d,), std(d,)
(data - mean) / std               # (d,) auto-broadcasts over n rows
```

In [2]:
import numpy as np

# Views vs copies
arr = np.arange(12).reshape(3, 4)
view = arr.T                    # transpose is a view
copy = arr[[0, 2]]              # fancy indexing is always a copy
print("Transpose is view:", np.shares_memory(arr, view))   # True
print("Fancy index is view:", np.shares_memory(arr, copy)) # False

# Strides demo
m = np.zeros((3, 4, 5), dtype=np.float64)  # 8 bytes per element
print(f"\nShape {m.shape}, strides {m.strides}")
# Strides = (4*5*8, 5*8, 8) = (160, 40, 8)
# → axis 2 (last) has smallest stride → innermost loop

def best_loop_order(strides, names):
    return [n for _, n in sorted(zip(strides, names), reverse=True)]

order = best_loop_order((600, 40, 8, 200), ('i','j','k','l'))
print(f"\nFor strides (600,40,8,200): outer→inner = {' → '.join(order)}")

# Broadcasting shape calculation
def broadcast_shape(shape_a, shape_b):
    la, lb = len(shape_a), len(shape_b)
    if la < lb: shape_a = (1,)*(lb-la) + tuple(shape_a)
    else:       shape_b = (1,)*(la-lb) + tuple(shape_b)
    result = []
    for da, db in zip(shape_a, shape_b):
        if da == db: result.append(da)
        elif da == 1: result.append(db)
        elif db == 1: result.append(da)
        else: return None
    return tuple(result)

print(f"\n(100,1,6,3) + (100,1,3) → {broadcast_shape((100,1,6,3),(100,1,3))}")

Transpose is view: True
Fancy index is view: False

Shape (3, 4, 5), strides (160, 40, 8)

For strides (600,40,8,200): outer→inner = i → l → j → k

(100,1,6,3) + (100,1,3) → (100, 100, 6, 3)


---
# 3. Cache, Memory Hierarchy & Hardware
### *Fast Python Ch. 6*

---

## Memory Hierarchy — Sizes and Speeds

| Level | Typical Size | Access Time | Implication |
|---|---|---|---|
| L1 cache | 32–256 KB | ~2 ns | Fastest — keep hot data here |
| L2 cache | 256 KB–1 MB | ~5 ns | |
| L3 cache | 6–20 MB | ~30 ns | Shared across cores |
| RAM | GBs | ~100 ns | 50× slower than L1 |
| SSD | TBs | ~50 µs | 25,000× slower than L1 |
| HDD | TBs | ~5 ms | 2,500,000× slower than L1 |

```bash
# Read cache sizes on HPC:
lscpu | grep cache
# L1d cache: 32K  L2 cache: 1024K  L3 cache: 19712K
```

---

## Row vs Column Access (NumPy is row-major by default)

```
Matrix in memory: [row0col0, row0col1, row0col2, row1col0, row1col1, ...]

mat[0, :]  → sequential memory → cache-friendly  → FAST ✅
mat[:, 0]  → strided memory    → cache misses    → SLOW ❌
```

**When does performance diverge?** When the matrix no longer fits in L1 cache.
- Array fits in L1 → row and column access equally fast
- Array larger than L1 → column access hits L2, then L3, then RAM

---

## The Counterintuitive Compression Insight (*Fast Python* Ch. 6)

> "Sometimes it is faster to work with **compressed data** than with uncompressed data,  
> even if we must decompress it first."

**Why?** If the compressed block fits in CPU cache but raw data doesn't:
- CPU loads compressed block from RAM once into cache
- Decompresses using idle CPU cycles (CPU is much faster than RAM)
- Net result: fewer cache misses, less time waiting for RAM

**Application:** Blosc compression, Zarr compressed chunks.

---

## Measuring MFLOP/s

$$\text{MFLOP/s} = \frac{n_{\text{elements}} \times 1 \text{ FLOP}}{t_{\text{seconds}} \times 10^6}$$

```python
# Size in KiB for cache plots:
size_kib = n_rows * n_cols * dtype_bytes / 1024
# float64 = 8 bytes, float32 = 4, uint8 = 1
```

In [3]:
from time import perf_counter as time
import numpy as np

def measure_row_col_mflops(SIZE, n_repeat=500):
    mat = np.random.rand(SIZE, SIZE)  # float64 = 8 bytes
    size_kib = SIZE * SIZE * 8 / 1024

    t = time()
    for _ in range(n_repeat): mat[0, :] * 2
    t_row = (time() - t) / n_repeat

    t = time()
    for _ in range(n_repeat): mat[:, 0] * 2
    t_col = (time() - t) / n_repeat

    mf_row = SIZE / t_row / 1e6  # SIZE FLOPs (one multiply each)
    mf_col = SIZE / t_col / 1e6
    return size_kib, mf_row, mf_col

print(f"{'Size':>8} {'KiB':>10} {'Row MFLOP/s':>13} {'Col MFLOP/s':>13} {'Ratio':>7}")
print("-" * 60)
for n in [100, 500, 2000]:
    kb, row, col = measure_row_col_mflops(n)
    print(f"{n:>8} {kb:>10.1f} {row:>13.1f} {col:>13.1f} {row/col:>7.1f}x")
print("\nKey: ratio grows as array exceeds cache sizes")

    Size        KiB   Row MFLOP/s   Col MFLOP/s   Ratio
------------------------------------------------------------
     100       78.1         211.2         213.6     1.0x
     500     1953.1         990.6         746.4     1.3x
    2000    31250.0        1180.5         604.1     2.0x

Key: ratio grows as array exceeds cache sizes


---
# 4. Data Compression — Blosc & Efficient Storage
### *Fast Python Ch. 6*

---

## When to use Blosc (compressed storage)

| Data type | Compresses well? | Use Blosc? |
|---|---|---|
| Zeros / constant values | ✅ Excellent | Yes — often 1000×+ smaller |
| Structured/tiled patterns | ✅ Good | Yes |
| Random data | ❌ Poor | No — overhead not worth it |

**Rule:** Use Blosc when the compressed file fits in cache but raw data doesn't.
If data is random, compression won't help and may hurt (overhead cost).

## Compression algorithms (cname parameter)

| Algorithm | Speed | Ratio | Use when |
|---|---|---|---|
| `lz4` | Very fast | Moderate | Default — balanced |
| `zstd` | Slower | Better | Need smaller files, have CPU time |
| `blosclz` | Fastest | Lower | Speed is critical |

## Blosc API (supporting functions from exercises)

```python
import blosc
import numpy as np

# Write compressed
b_arr = blosc.pack_array(arr, cname='lz4')  # compress
with open('file.bl', 'wb') as f:
    f.write(b_arr)

# Read compressed
with open('file.bl', 'rb') as f:
    b_arr = f.read()
arr = blosc.unpack_array(b_arr)  # decompress

# os.sync() flushes OS I/O buffers — needed for fair timing comparisons
import os; os.sync()
```

## Key insight from exercise
- **Zeros array**: Blosc read is much faster than numpy (tiny file = fast I/O)
- **Random array**: numpy read may be faster (no compression benefit, but pay decompression cost)
- **Crossover point**: depends on how compressible the data is and disk speed vs CPU speed

---
# 5. Pandas — dtype Optimisation & Indexing
### *Fast Python Ch. 7*

---

## dtype Recoding to Reduce Memory

### Step-by-step for each column:
1. **Object (string)?** → check if dates → `datetime64`; few unique values → `category`
2. **Integer?** → find [min, max] → pick smallest int type that fits
3. **Float?** → `float32` if precision allows

### Integer type ranges

| dtype | Bytes | Min | Max |
|---|---|---|---|
| `int8` | 1 | -128 | 127 |
| `uint8` | 1 | 0 | 255 |
| `int16` | 2 | -32,768 | 32,767 |
| `uint16` | 2 | 0 | 65,535 |
| `int32` | 4 | -2.1B | 2.1B |
| `uint32` | 4 | 0 | 4.3B |
| `int64` | 8 | -9.2e18 | 9.2e18 |

### Exam example (Exam 2024 Q15)
```python
df['date']     = pd.to_datetime(df['date'])    # object → datetime64
df['location'] = df['location'].astype('category')  # 8 unique values → category
df['mach_id']  = df['mach_id'].astype('int16') # [-1, 5730] fits in int16
df['units']    = df['units'].astype('int32')   # [932, 68837] fits in int32
```

---

## Speed Up Row Extraction — Sorted Index

**Problem:** Extracting rows by date is slow (O(n) scan)

**Solution:** Set date as sorted index → binary search O(log n)

```python
# Overhead to build once; pays off when many queries are made
df = df.set_index('date').sort_index()
rows = df.loc['2024-05-28']   # fast!
```

---

## Chunked Processing — Memory Budget

```python
import numpy as np

def max_chunk_rows(dtypes, budget_mb):
    bytes_per_row = sum(np.dtype(d).itemsize for d in dtypes)
    return (budget_mb * 1_000_000) // bytes_per_row

# Example: sensor_id(uint32), timestamp(uint64), power(float64), budget=200MB
max_chunk_rows([np.uint32, np.uint64, np.float64], 200)
# bytes_per_row = 4+8+8 = 20
# max rows = 200,000,000 / 20 = 10,000,000
```

In [4]:
import numpy as np

# ── Integer type range checker ────────────────────────────────────
def smallest_int_dtype(val_min, val_max):
    for dt in [np.int8, np.uint8, np.int16, np.uint16, np.int32, np.uint32]:
        info = np.iinfo(dt)
        if info.min <= val_min and val_max <= info.max:
            return dt.__name__, np.dtype(dt).itemsize
    return 'int64', 8

print("dtype recoding decisions:")
tests = [(-1, 5730, 'mach_id'), (932, 68837, 'units'), (0, 42, 'version'), (0, 255, 'small_int')]
for vmin, vmax, name in tests:
    dtype, nbytes = smallest_int_dtype(vmin, vmax)
    print(f"  {name:<12} [{vmin:>8}, {vmax:>8}] → {dtype:<8} ({nbytes} bytes vs 8 for int64)")

print()

# ── Chunk size calculator ────────────────────────────────────────
def max_chunk_rows(dtypes, budget_mb):
    bpr = sum(np.dtype(d).itemsize for d in dtypes)
    return budget_mb * 1_000_000 // bpr

print("Max chunk rows:")
print(f"  Sensor data (uint32+uint64+float64), 200 MB: {max_chunk_rows([np.uint32,np.uint64,np.float64],200):,}")
print(f"  3×int64 columns, 24 MB: {max_chunk_rows([np.int64]*3, 24):,}")

dtype recoding decisions:
  mach_id      [      -1,     5730] → int16    (2 bytes vs 8 for int64)
  units        [     932,    68837] → int32    (4 bytes vs 8 for int64)
  version      [       0,       42] → int8     (1 bytes vs 8 for int64)
  small_int    [       0,      255] → uint8    (1 bytes vs 8 for int64)

Max chunk rows:
  Sensor data (uint32+uint64+float64), 200 MB: 10,000,000
  3×int64 columns, 24 MB: 1,000,000


---
# 6. Zarr — Chunked On-Disk Arrays
### *Fast Python Ch. 8*

---

## What is Zarr?
Zarr stores large N-dimensional arrays on disk in **chunks**. When you access data, only the relevant chunks are loaded into memory. This allows processing arrays **larger than RAM**.

```python
import zarr

# Open (read-only)
x = zarr.open('myarray.zarr', mode='r')

# Accessing data loads only needed chunks from disk
row = x[5, :]       # loads chunk(s) containing row 5
col = x[:, 3]       # loads chunk(s) containing column 3
```

---

## Choosing the Best Chunk Shape

**Goal:** Minimise the number of chunk reads per access.

Match chunk shape to your access pattern:

| Access pattern | Best chunk shape | Why |
|---|---|---|
| Full rows: `x[i, :]` | `(1, ncols)` | Each row = 1 chunk read |
| Full cols: `x[:, j]` | `(nrows, 1)` | Each col = 1 chunk read |
| 2D blocks | `(n, n)` square | Balanced |

### Exam example (Re-exam Q5)
Matrix `1000×100000`, reading full columns `x[:, j]`:

| Chunk | Reads per column | |
|---|---|---|
| `10×10000` | 1000/10 = 100 | ❌ |
| `100×1000` | 1000/100 = 10 | ❌ |
| `1000×100` | 1000/1000 = **1** | ✅ Best |

### Memory per chunk
$$\text{chunk memory} = \text{chunk\_rows} \times \text{chunk\_cols} \times \text{itemsize}$$

```python
# Example: chunk (1000, 100) with float64 (8 bytes)
memory = 1000 * 100 * 8   # = 800,000 bytes = 800 KB
```

---

## np.memmap — Memory-Mapped Files

```python
import numpy as np

# Maps a file to memory without loading it — uses virtual memory
x = np.memmap('bigarray.raw', mode='r', dtype='uint8', shape=10_000_000_000)

# Only the accessed data is loaded into RAM
y = np.array(x[::100_000])  # copies 100,000 elements into RAM
# Memory used = 100,000 × 1 byte = 100 KB  (NOT 10 GB!)
```

In [5]:
import math

def zarr_reads_per_access(array_shape, chunk_shape, access_shape):
    """How many chunk reads are needed for one access?"""
    return math.prod(
        math.ceil(a / c) for a, c in zip(access_shape, chunk_shape)
    )

# Re-exam Q5: 1000×100000 array, read full columns x[:,j]
print("=== Re-exam Q5: Read full columns from 1000×100000 array ===")
access = (1000, 1)  # full column
for chunk, label in [((10,10000),'a'), ((100,1000),'b'), ((1000,100),'c')]:
    n = zarr_reads_per_access((1000,100000), chunk, access)
    ok = "✅ BEST" if n == 1 else "❌"
    print(f"  ({label}) chunk={chunk}: {n} reads per column  {ok}")

print()
# F25 Q20: 1024×1024 array, sum each row: a[i]
print("=== F25 Q20: Sum each row of 1024×1024 array ===")
access_row = (1, 1024)
for chunk, label in [((1,1024),'A'), ((1024,1),'B'), ((32,32),'C')]:
    n = zarr_reads_per_access((1024,1024), chunk, access_row)
    ok = "✅ BEST" if n == 1 else "❌"
    print(f"  ({label}) chunk={chunk}: {n} reads per row  {ok}")

=== Re-exam Q5: Read full columns from 1000×100000 array ===
  (a) chunk=(10, 10000): 100 reads per column  ❌
  (b) chunk=(100, 1000): 10 reads per column  ❌
  (c) chunk=(1000, 100): 1 reads per column  ✅ BEST

=== F25 Q20: Sum each row of 1024×1024 array ===
  (A) chunk=(1, 1024): 1 reads per row  ✅ BEST
  (B) chunk=(1024, 1): 1024 reads per row  ❌
  (C) chunk=(32, 32): 32 reads per row  ❌


---
# 7. Amdahl's Law
### *Advanced Python Programming Ch. 9*

---

## The Formula

Let **B = serial fraction** (fraction of program that cannot be parallelised).

With N processors:
$$S(N) = \frac{1}{B + \frac{1-B}{N}} = \frac{T(1)}{T(N)}$$

Note: The course uses **F = parallel fraction** = 1 - B, so:
$$S(p) = \frac{1}{(1-F) + F/p}$$

---

## Key Implications (from the book)

1. **Speedup increases with more processors** — but with diminishing returns
2. **Upper limit exists**: as $N \to \infty$, $S \to 1/B = 1/(1-F)$
3. **Sequential overhead dominates** at scale — the serial part sets a hard ceiling
4. **Diminishing returns**: adding the 2nd processor helps more than the 100th

---

## All Variants

In [6]:
import numpy as np
import matplotlib.pyplot as plt

# Core functions
def S(F, p):        return 1 / ((1-F) + F/p)
def S_max(F):       return 1 / (1-F)
def F_from_S(s, p): return p*(1-1/s)/(p-1)      # from measured speedup
def F_from_plateau(plateau): return 1 - 1/plateau  # from speedup plot
def F_from_times(T_serial, T_parallel): return T_parallel/(T_serial+T_parallel)

print("=== ALL VARIANTS ===")
print()

# Variant A: S(p) from F and p
print(f"A: F=0.8, p=8  → S = {S(0.8, 8):.3f}")

# Variant B: Max speedup
print(f"B: F=0.8       → S_max = {S_max(0.8):.1f}")

# Variant C: Should they pursue? (compare to target)
F, target, p_best = 0.8, 4, 8
s_best = S(F, p_best)
print(f"C: F={F}, target={target}x, {p_best} cores → S={s_best:.2f} → {'YES ✅' if s_best>=target else 'NO ❌ — cannot reach target'}")

# Variant D: Serial time from parallel time
T_p, F_d, p_d = 10, 0.8, 4  # 10 min on 4 cores, F=0.8
T_1 = T_p * S(F_d, p_d)
print(f"D: T(4)={T_p}min, F={F_d} → S(4)={S(F_d,p_d):.2f} → T(1)={T_1:.1f} min")

# Variant E: New time after reducing serial part
reduction = 3  # serial part reduced by 3 min
print(f"E: T(4) was {T_p}min. Serial reduced by {reduction}min → new T(4) = {T_p-reduction}min")

# Variant F: F from serial/parallel times
T_s, T_par = 20, 100
print(f"F: T_serial={T_s}s, T_parallel={T_par}s → F={F_from_times(T_s,T_par):.4f}")

# Variant G: F from measured speedup
s_measured, p_m = 2.5, 3
F_est = F_from_S(s_measured, p_m)
print(f"G: S({p_m})={s_measured} → F={F_est:.3f}, S_max={S_max(F_est):.1f}")

# Variant H: F from speedup plot plateau
plateau = 5
print(f"H: Plateau={plateau} → F=1-1/{plateau}={F_from_plateau(plateau):.2f}")

=== ALL VARIANTS ===

A: F=0.8, p=8  → S = 3.333
B: F=0.8       → S_max = 5.0
C: F=0.8, target=4x, 8 cores → S=3.33 → NO ❌ — cannot reach target
D: T(4)=10min, F=0.8 → S(4)=2.50 → T(1)=25.0 min
E: T(4) was 10min. Serial reduced by 3min → new T(4) = 7min
F: T_serial=20s, T_parallel=100s → F=0.8333
G: S(3)=2.5 → F=0.900, S_max=10.0
H: Plateau=5 → F=1-1/5=0.80


---
# 8. The GIL — Threading vs Multiprocessing
### *Advanced Python Programming Ch. 22 | Fast Python Ch. 3*

---

## What is the GIL?

The **Global Interpreter Lock** is a single lock on the entire Python interpreter.  
Any Python instruction that wants to execute must first acquire the GIL.

> *"The GIL is a solution to making sure that race conditions will not occur with  
> regard to Python's reference counting."* — Advanced Python Programming Ch. 22

---

## The Core Problem

```python
# CPU-bound threading: both threads fight for GIL → no speedup!
import threading
def countdown(n):
    while n > 0: n -= 1   # pure Python → GIL held the whole time

# Sequential: 2.80s
# Threaded (2 threads): 2.74s  ← barely faster! (sometimes even slower)
```

---

## Decision Table

| Code type | GIL released? | Best approach |
|---|---|---|
| Pure Python `for` loop | ❌ No | **Multiprocessing** |
| NumPy operations | ✅ Yes (C backend) | Threading OK |
| Numba `@jit(nogil=True)` | ✅ Yes | Threading OK |
| Numba `@jit` without nogil | ❌ No | Multiprocessing |
| I/O-bound (file/network) | ✅ Yes (I/O wait) | Threading OK |

### 3-question test
```
Q1: Is it a pure Python loop (no NumPy/Numba)?  YES → Multiprocessing
Q2: Does it use NumPy or Numba with nogil=True?  YES → Threading is fine
Q3: Is it I/O-bound?                             YES → Threading is fine
Otherwise:                                              Multiprocessing
```

---

## How to Work with the GIL (from the book)

1. **Use multiprocessing** — each process has its own GIL, truly parallel
2. **Use C extensions** (NumPy, Numba) — they release the GIL during computation
3. **Use Cython with `nogil`** — explicitly release GIL in Cython code
4. **Use async for I/O** — GIL is released during I/O waits anyway

```python
# Numba: release GIL so threading works
from numba import jit

@jit(nopython=True, nogil=True)  # ← nogil=True releases GIL
def simulate_single(x0, n, step):
    x = x0
    for i in range(n):           # sequential — each step depends on previous
        x = x + step * np.cos(x) / (np.sin(5*x) + 2.5)
    return x

# Can now parallelise the OUTER loop with threads:
from multiprocessing.pool import ThreadPool
with ThreadPool(m) as pool:
    results = pool.starmap(simulate_single, [(x0, n, step) for x0 in x0s])
```

---
# 9. Parallel Reduction
### *Advanced Python Programming Ch. 14*

---

## What is a Reduction Operator?

An operator that reduces an array to a single value by combining elements pairwise.

For an operator `⊕` to work in **parallel reduction**, it must be:
- **Commutative**: `a ⊕ b = b ⊕ a`
- **Associative**: `(a ⊕ b) ⊕ c = a ⊕ (b ⊕ c)`

---

## Binary Tree Reduction

```
[1, 4, 8, 3, 2, 5]

Step 1 (3 processes): (1+4)  (8+3)  (2+5)  →  [5, 11, 7]
Step 2 (1 process):  (5+11)  (7)    →  [16, 7]
Step 3 (1 process):  (16+7)  →  [23]

log₂(6) ≈ 3 steps instead of 5 sequential steps
```

**Speedup of parallel reduction vs sequential:**
$$\text{speedup} = \frac{n}{\log_2(n)}$$

**Speedup vs row-sum approach** (from re-exam Q18):
$$\text{speedup} = \frac{2n}{2\log_2(n)} = \frac{n}{\log_2(n)}$$

---

## Valid and Invalid Reduction Operators

In [7]:
import math

def check_associative(f, a, b, c, name):
    left  = f(f(a, b), c)
    right = f(a, f(b, c))
    ok = abs(left - right) < 1e-9 if isinstance(left, float) else left == right
    print(f"{'✅' if ok else '❌'} {name:<20} f(f(a,b),c)={left}  f(a,f(b,c))={right}  {'OK' if ok else 'NOT ASSOCIATIVE — cannot use in reduction'}")

a, b, c = 1, 2, -3
print("Checking associativity with a=1, b=2, c=-3:\n")
check_associative(lambda x,y: x+y,     a,b,c, "x + y (sum)")
check_associative(lambda x,y: x*y,     a,b,c, "x * y (product)")
check_associative(lambda x,y: max(x,y),a,b,c, "max(x,y)")
check_associative(lambda x,y: abs(x+y),a,b,c, "abs(x+y)")
check_associative(lambda x,y: x-y,     a,b,c, "x - y")

# Set intersection
A,B,C = {1,2,3},{2,3,4},{3,4,5}
left  = (A&B)&C
right = A&(B&C)
print(f"✅ Set intersection   (A∩B)∩C={left}  A∩(B∩C)={right}  OK")

print()
print("Parallel reduction speedup over sequential:")
for n in [100, 1000, 10000, 1000000]:
    print(f"  n={n:>10,}: n/log₂(n) = {n/math.log2(n):.1f}x faster")

Checking associativity with a=1, b=2, c=-3:

✅ x + y (sum)          f(f(a,b),c)=0  f(a,f(b,c))=0  OK
✅ x * y (product)      f(f(a,b),c)=-6  f(a,f(b,c))=-6  OK
✅ max(x,y)             f(f(a,b),c)=2  f(a,f(b,c))=2  OK
❌ abs(x+y)             f(f(a,b),c)=0  f(a,f(b,c))=2  NOT ASSOCIATIVE — cannot use in reduction
❌ x - y                f(f(a,b),c)=2  f(a,f(b,c))=-4  NOT ASSOCIATIVE — cannot use in reduction
✅ Set intersection   (A∩B)∩C={3}  A∩(B∩C)={3}  OK

Parallel reduction speedup over sequential:
  n=       100: n/log₂(n) = 15.1x faster
  n=     1,000: n/log₂(n) = 100.3x faster
  n=    10,000: n/log₂(n) = 752.6x faster
  n= 1,000,000: n/log₂(n) = 50171.7x faster


---
# 10. Multiprocessing Patterns
### *Advanced Python Programming Ch. 7 & 13 | Fast Python Ch. 3*

---

## Three Parallelisation Patterns

### Pattern 1: `apply_async` — most flexible
```python
import multiprocessing

with multiprocessing.Pool(n_proc) as pool:
    results = [pool.apply_async(func, (arg,)) for arg in data]
    output  = [r.get() for r in results]   # blocks until done
```

### Pattern 2: `pool.map` — simplest
```python
with multiprocessing.Pool(n_proc) as pool:
    output = pool.map(func, data, chunksize=len(data)//n_proc)
```

### Pattern 3: Chunked (best for heavy workloads)
```python
def process_chunk(chunk_size):
    return sum(func(x) for x in range(chunk_size))

chunk_size = n_samples // n_proc
with multiprocessing.Pool(n_proc) as pool:
    results = [pool.apply_async(process_chunk, (chunk_size,)) for _ in range(n_proc)]
    total   = sum(r.get() for r in results)
```

---

## Static vs Dynamic Scheduling

| | Static | Dynamic |
|---|---|---|
| Work division | Upfront, equal chunks | On-demand from queue |
| Overhead | Low | Higher |
| Best for | Equal task durations (low StdDev) | Variable durations (high StdDev) |
| Risk | Load imbalance → workers idle | None |

```python
# Static: large chunksize → divide upfront
pool.map(func, tasks, chunksize=len(tasks)//n_workers)

# Dynamic: chunksize=1 → pick one at a time
pool.map(func, tasks, chunksize=1)
for result in pool.imap_unordered(func, tasks, chunksize=1):
    process(result)
```

**Mandelbrot example:** Points inside set → 100 iterations; outside → 1-5 iterations.
Static scheduling → fast workers idle waiting for slow ones. Dynamic (small chunks) → F goes from 0.945 to 0.98.

---

## Can the Loop be Parallelised?

```python
# YES — iterations are independent
for scene in all_scenes:
    frame = render_scene(scene)   # each scene is independent ✅

# NO — each iteration depends on the previous
for i in range(n):
    scene = advance_scene(scene, dt)  # depends on previous scene ❌

# NO — simulation steps are sequential
for i in range(n):
    x = x + step * np.cos(x)  # each x depends on previous ❌
# → Can parallelise the OUTER loop (over different initial conditions)
```

In [11]:
from concurrent.futures import ThreadPoolExecutor
import random


def sample_pi():
    x, y = random.uniform(-1, 1), random.uniform(-1, 1)
    return 1 if x**2 + y**2 <= 1 else 0


def sample_multiple(n):
    return sum(sample_pi() for _ in range(n))


samples = 100_000
n_threads = 4


# Pattern 1
with ThreadPoolExecutor(max_workers=n_threads) as executor:
    results = list(executor.map(lambda _: sample_pi(), range(samples)))

pi_1 = 4.0 * sum(results) / samples
print(f"Pattern 1: π ≈ {pi_1:.4f}")


# Pattern 2
chunk = samples // n_threads

with ThreadPoolExecutor(max_workers=n_threads) as executor:
    results = list(executor.map(sample_multiple, [chunk] * n_threads))

pi_2 = 4.0 * sum(results) / samples
print(f"Pattern 2: π ≈ {pi_2:.4f}")

Pattern 1: π ≈ 3.1361
Pattern 2: π ≈ 3.1418


---
# 11. GPU & CUDA
### *Fast Python Ch. 9*

---

## GPU Architecture (why it's different from CPU)

| | CPU | GPU |
|---|---|---|
| Cores | Few (4–64) | Thousands (simple) |
| Best for | Sequential, complex logic | Massively parallel, simple ops |
| Memory | Shared with RAM | **Separate** from CPU RAM |
| Programming | Straightforward | Requires special kernel code |

> *"GPU memory is separated from main memory. There is the problem of  
> transferring data between main memory and GPU memory."* — Fast Python Ch. 1

---

## CUDA Kernels with Numba

```python
from numba import cuda

@cuda.jit
def my_kernel(array, out):
    i, j = cuda.grid(2)           # 2D thread index
    if i < array.shape[0] and j < array.shape[1]:
        out[i, j] = array[i, j] * 2

# Launch: must specify blocks per grid and threads per block
threadsperblock = (32, 32)        # 32×32 = 1024 threads per block
import math
blockspergrid = (
    math.ceil(H / 32),
    math.ceil(W / 32),
)
my_kernel[blockspergrid, threadsperblock](array, out)
```

### Number of thread blocks
$$\text{blocks per dim} = \left\lceil \frac{\text{output size}}{\text{threads per block}} \right\rceil$$

---

## Memory Transfers — the main bottleneck

NumPy arrays passed to CUDA kernels are **automatically transferred both ways**.

```python
# Inefficient: both arrays transferred every call
for x in images:
    kernel[blocks, threads](x_numpy, y_numpy)  # HtoD + DtoH both arrays!

# Efficient: keep output on GPU
y_gpu = cuda.device_array(shape, dtype)
for x in images:
    x_gpu = cuda.to_device(x)         # 1 HtoD per image
    kernel[blocks, threads](x_gpu, y_gpu)  # no auto-transfer!
result = y_gpu.copy_to_host()         # 1 DtoH at the end
```

### Total GPU pipeline time
$$T_{GPU} = T_{kernel} + T_{HtoD} + T_{DtoH}$$

---

## Array Layout for GPU — Coalesced Memory Access

**Warp** = 32 threads executing the same instruction simultaneously.

For coalesced access: adjacent threads must access **adjacent memory**.

In a 2D block `(rows, cols)`, threads in a warp vary along the **column dimension (j)**.
→ The j-axis should have the smallest stride → **j should be the last axis**.

| Context | Layout |
|---|---|
| CPU: inner loop over channels | `H × W × C` (channels **last**) |
| GPU: all threads access same channel | `C × H × W` (channels **first**) |

---

## Reading nsys Profiler Output

```
** CUDA GPU Kernel Summary:
  100.0%   0.5000s   1   conv_channels_kernel

** GPU MemOps (by Time):
  83.3%    2.5000s   2   [CUDA memcpy HtoD]   ← CPU → GPU
  16.7%    0.5000s   1   [CUDA memcpy DtoH]   ← GPU → CPU

** GPU MemOps (by Size):
  25000 MB  2  [CUDA memcpy HtoD]
   1000 MB  1  [CUDA memcpy DtoH]
```

Transfer speed: `total_MB / total_seconds` = 25000 / 2.5 = **10,000 MB/s = 10 GB/s**

Total GPU time: 0.5 + 2.5 + 0.5 = **3.5 s** vs CPU 7 s → **2× faster**

In [12]:
import math

def thread_blocks(output_shape, threads_per_block):
    blocks = tuple(math.ceil(o/t) for o,t in zip(output_shape, threads_per_block))
    print(f"Output {output_shape}, TPB {threads_per_block} → blocks {blocks}")
    return blocks

print("=== Thread block calculations ===")
thread_blocks((200, 200), (16, 16))   # F25 Q13 → 13×13
thread_blocks((500, 500), (32, 32))   # → 16×16

def nsys_analysis(kernel_s, htod_s, htod_mb, dtoh_s, dtoh_mb, cpu_s=None):
    T_gpu = kernel_s + htod_s + dtoh_s
    print(f"\nKernel: {kernel_s:.3f}s | HtoD: {htod_s:.3f}s ({htod_mb:.0f}MB) | DtoH: {dtoh_s:.3f}s ({dtoh_mb:.0f}MB)")
    print(f"Total GPU time: {T_gpu:.3f}s")
    print(f"HtoD speed: {htod_mb/htod_s/1000:.1f} GB/s")
    if cpu_s:
        print(f"Speedup vs CPU ({cpu_s}s): {cpu_s/T_gpu:.1f}x")
    times = {'kernel':kernel_s,'HtoD':htod_s,'DtoH':dtoh_s}
    print(f"Bottleneck: {max(times, key=times.get)} ({max(times.values()):.3f}s)")

print("\n=== Exam 2024 Q13/Q14 ===")
nsys_analysis(0.5, 2.5, 25000, 0.5, 1000, cpu_s=7.0)

=== Thread block calculations ===
Output (200, 200), TPB (16, 16) → blocks (13, 13)
Output (500, 500), TPB (32, 32) → blocks (16, 16)

=== Exam 2024 Q13/Q14 ===

Kernel: 0.500s | HtoD: 2.500s (25000MB) | DtoH: 0.500s (1000MB)
Total GPU time: 3.500s
HtoD speed: 10.0 GB/s
Speedup vs CPU (7.0s): 2.0x
Bottleneck: HtoD (2.500s)


---
# 12. Numba JIT
### *Fast Python Appendix B*

---

## What is Numba?

Numba compiles Python functions to optimised machine code using LLVM — **at first call**.  
It's particularly effective for numerical loops that NumPy cannot vectorise easily.

> *"For real-world problems, I recommend Numba. Cython, with its extra hurdles,  
> allows us to dig deeper in understanding what is going on."* — Fast Python App. B note

---

## Usage

```python
from numba import jit
import numpy as np

@jit(nopython=True)           # compile to machine code — no Python fallback
def fast_sum(arr):
    s = 0.0
    for x in arr:
        s += x
    return s

@jit(nopython=True, nogil=True)  # also releases GIL → threading works!
def simulate_step(x, step):
    return x + step * np.cos(x) / (np.sin(5*x) + 2.5)
```

---

## Key Rules for the Exam

| Rule | Detail |
|---|---|
| First call compiles | Subsequent calls are fast — warm-up cost on first call |
| `nopython=True` | Full compilation — crashes if Python objects used |
| `nogil=True` | Releases GIL → **multithreading works** with this function |
| Sequential loops | Cannot parallelise if each step depends on previous |
| Independent outer loop | **Can** parallelise the outer loop with threads |

---

## Exam Q19 Pattern — Where to Parallelise

```python
@jit(nopython=True, nogil=True)
def simulate_single(x0, n, step):
    x = x0
    for i in range(n):           # ❌ sequential — each step depends on x
        x = x + step * np.cos(x) / (np.sin(5*x) + 2.5)
    return x

def simulate(n, x0s, step):     # ✅ parallelise HERE — each x0 is independent
    from multiprocessing.pool import ThreadPool  # threading OK: nogil=True
    with ThreadPool(len(x0s)) as pool:
        return pool.starmap(simulate_single, [(x0, n, step) for x0 in x0s])
    # Expected speedup: up to m× (where m = len(x0s))
    # Does NOT depend on n (n only affects how long each call takes)
```

---

# SUMMARY — Complete Formula & Decision Reference

In [13]:
import numpy as np, math

print("="*65)
print("COMPLETE FORMULA REFERENCE")
print("="*65)

print("""
── AMDAHL'S LAW ──────────────────────────────────────────────────
  S(p) = 1 / [(1-F) + F/p]        speedup on p cores
  S_max = 1/(1-F)                  max speedup (p→∞)
  F = 1 - 1/plateau                from speedup plot
  F = p*(1-1/S)/(p-1)              from measured S on p cores
  F = T_parallel/(T_s+T_p)         from serial/parallel times
  T_new_p = T_old_p - reduction    after reducing serial part
  T_1 = T_p × S(F,p)              serial time from parallel time

── LSF MEMORY ────────────────────────────────────────────────────
  mem_per_core = total_GB / n_cores
  total = mem_per_core × n_cores
  done()  = wait until ALL succeed (EXIT breaks it → never starts)
  ended() = wait until ALL finish (DONE or EXIT)

── MFLOP/s ───────────────────────────────────────────────────────
  FLOP/s = (FLOPs/iter × iters) / total_time_s
  FLOPs:  +,-,*,/,sqrt,sin,cos = 1 each

── CHUNK SIZE ────────────────────────────────────────────────────
  bytes_per_row = sum(itemsize for each column)
  max_rows = budget_MB × 1e6 / bytes_per_row

── CUDA BLOCKS ───────────────────────────────────────────────────
  blocks_per_dim = ceil(output_size / threads_per_block)
  T_GPU = T_kernel + T_HtoD + T_DtoH
  Transfer speed (GB/s) = total_MB / total_s / 1000

── PARALLEL REDUCTION ────────────────────────────────────────────
  Must be ASSOCIATIVE: f(f(a,b),c) == f(a,f(b,c))
  Speedup vs sequential: n / log₂(n)

── ZARR CHUNKS ───────────────────────────────────────────────────
  Reads per access = ceil(access_dim / chunk_dim) per dimension
  Goal: minimise reads → chunk matches access pattern
""")

print("── DTYPE SIZES ───────────────────────────────────────────────────")
for dt in [np.int8,np.uint8,np.int16,np.uint16,np.int32,np.uint32,np.float32,np.float64]:
    info = np.iinfo(dt) if 'int' in dt.__name__ else np.finfo(dt)
    print(f"  {dt.__name__:<10} {np.dtype(dt).itemsize} bytes")

print("""
── DECISION TREE ─────────────────────────────────────────────────
  Slow code? → Profile first (cProfile, line_profiler)
  Loop parallelisable? → iterations independent? → YES → parallelise
  Pure Python loop? → multiprocessing
  NumPy/Numba nogil? → threading OK
  Task times vary? → dynamic scheduling
  Cache: CPU inner loop → smallest stride axis innermost
  GPU layout: threads access same channel → channels first (C×H×W)
  Zarr: chunk shape matches your access pattern
  Pandas: set sorted index for repeated date queries
""")

COMPLETE FORMULA REFERENCE

── AMDAHL'S LAW ──────────────────────────────────────────────────
  S(p) = 1 / [(1-F) + F/p]        speedup on p cores
  S_max = 1/(1-F)                  max speedup (p→∞)
  F = 1 - 1/plateau                from speedup plot
  F = p*(1-1/S)/(p-1)              from measured S on p cores
  F = T_parallel/(T_s+T_p)         from serial/parallel times
  T_new_p = T_old_p - reduction    after reducing serial part
  T_1 = T_p × S(F,p)              serial time from parallel time

── LSF MEMORY ────────────────────────────────────────────────────
  mem_per_core = total_GB / n_cores
  total = mem_per_core × n_cores
  done()  = wait until ALL succeed (EXIT breaks it → never starts)
  ended() = wait until ALL finish (DONE or EXIT)

── MFLOP/s ───────────────────────────────────────────────────────
  FLOP/s = (FLOPs/iter × iters) / total_time_s
  FLOPs:  +,-,*,/,sqrt,sin,cos = 1 each

── CHUNK SIZE ────────────────────────────────────────────────────
  bytes_per_row = s